In [6]:
import os
import glob
import json
import math
from pathlib import Path

import cv2
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from diffusers import StableDiffusionXLControlNetPipeline, UniPCMultistepScheduler

from warp_and_controlnet_model import WarpAndControlNetModel

# ================== 配置 ==================
NPY_DIR = "/data/test_grid4_from_test_grid4///npy"  # cond 来源（通常是 3x3 单应矩阵）
CAMCONF_DIR = "/data/test_grid4_from_test_grid4/camconf"  # 文本来源
CKPT_DIR = "/data/mv_model_0_with_drop/checkpoint-21500/controlnet/"
OUT_DIR = "./results_grid4"
BASE_MODEL = "hotshotco/SDXL-512"
PROMPT_DEFAULT = "city topview from camera"
IMAGE_SIZE = 512

# 和训练 dataset 一致的 HSI 来源
HSI_PATH = "/data/try_small/result_new/Houston18.mat"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
os.makedirs(OUT_DIR, exist_ok=True)


# ================== 文本工具（与 dataset 一致） ==================
def load_config(path):
    with open(path, "r") as f:
        return json.load(f)


def generate_text(config):
    camera_height = config["location"][2]
    tilt_rad = math.pi / 2 - config["rotation_euler"][0]
    tilt_deg = math.degrees(tilt_rad)
    lens = config["data"]["lens"]
    conf_str = f"{{camera_height:{camera_height},tilt_deg:{tilt_deg:.1f},lens:{lens}}}"
    return f"city topview from camera, camera conf as following. {conf_str}"


def prompt_from_group_paths(paths):
    """
    paths 已按 num4 = 0,90,180,270 排好。
    这里优先用第一个（0度）对应的 camconf 生成文本。
    camconf 文件名规则：num1_num2_num3_num4_camconf.json
    """
    stem = Path(paths[0]).stem
    camconf_path = os.path.join(CAMCONF_DIR, f"{stem}_camconf.json")
    if not os.path.exists(camconf_path):
        # 兜底：组内找任意一个存在的 camconf
        for p in paths:
            s = Path(p).stem
            c = os.path.join(CAMCONF_DIR, f"{s}_camconf.json")
            if os.path.exists(c):
                camconf_path = c
                break
        else:
            return PROMPT_DEFAULT

    cfg = load_config(camconf_path)
    return generate_text(cfg)


# ================== 预处理工具 ==================
def resize_chw(chw: np.ndarray, h: int, w: int) -> np.ndarray:
    t = torch.from_numpy(chw).unsqueeze(0)  # (1,C,H,W)
    t = F.interpolate(t, size=(h, w), mode="bilinear", align_corners=False)
    return t.squeeze(0).numpy()


def stitch_2x2_np(cubes):
    # cubes 顺序: tl, tr, bl, br；每个为 (C,H,W)
    top = np.concatenate([cubes[0], cubes[1]], axis=2)
    bottom = np.concatenate([cubes[2], cubes[3]], axis=2)
    return np.concatenate([top, bottom], axis=1)


def _parse_name_tokens(stem: str):
    # 命名格式：num1_num2_num3_num4（num 可正可负）
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    try:
        nums = [int(p) for p in parts[:4]]
    except ValueError:
        return None
    num123 = f"{nums[0]}_{nums[1]}_{nums[2]}"
    num4 = nums[3]
    return num123, num4


def group_npy_files(npy_dir: str):
    """
    按文件名 num1_num2_num3_num4 分组：
    - num1_num2_num3 相同为一组（即 num123）
    - num4 按 0, 90, 180, 270 的大小顺序排列
    - 每组优先使用这四个角度；若不足 4 个则跳过
    """
    files = sorted(glob.glob(os.path.join(npy_dir, "*.npy")))
    groups = {}

    for p in files:
        stem = Path(p).stem
        parsed = _parse_name_tokens(stem)
        if parsed is None:
            continue
        num123, num4 = parsed
        groups.setdefault(num123, []).append((num4, p))

    target_angles = [0, 90, 180, 270]
    final = []

    for num123, items in groups.items():
        angle_to_path = {}
        for num4, p in sorted(items, key=lambda x: (x[0], x[1])):
            if num4 in target_angles and num4 not in angle_to_path:
                angle_to_path[num4] = p

        if all(a in angle_to_path for a in target_angles):
            ordered_paths = [angle_to_path[a] for a in target_angles]
            final.append((num123, ordered_paths))

    def key_fn(x):
        return tuple(int(t) for t in x[0].split("_"))

    final.sort(key=key_fn)
    return final


# ================== HSI + Warp（借鉴 dataset 逻辑） ==================
def init_hsi_and_warped(image_size=512):
    with h5py.File(HSI_PATH, "r") as f:
        hsi = np.array(f["ori_data"])  # (C,H,W)

    hsi_norm = hsi / (hsi.max((1, 2), keepdims=True) + 1e-8)
    hsi_norm = hsi_norm.transpose(1, 2, 0)  # (H,W,C)

    P = np.array([
        [4.80769231e-02, 2.55899318e+00, 4.44000000e+02],
        [2.55528846e+00, -4.82433141e-02, 1.21000000e+02],
        [0.00000000e+00, 0.00000000e+00, 1.00000000e+00],
    ], dtype=np.float32)

    warped_hsi = cv2.warpPerspective(
        hsi_norm, P, (4096, 1024),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    sub = image_size // 2
    scale = sub / 512.0
    scale_mat = np.array([
        [scale, 0.0, 0.0],
        [0.0, scale, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=np.float32)

    return warped_hsi, scale_mat


def warp_hsi_from_homography(H_mat: np.ndarray, warped_hsi: np.ndarray, scale_mat: np.ndarray, out_size: int):
    M = scale_mat @ H_mat
    warped = cv2.warpPerspective(
        warped_hsi, M, (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return warped.transpose(2, 0, 1)


def load_cond48_from_group(paths, warped_hsi, scale_mat, image_size=512):
    sub = image_size // 2
    cubes = []

    for p in paths:
        arr = np.load(p)

        if arr.ndim == 2 and arr.shape == (3, 3):
            cube = warp_hsi_from_homography(arr.astype(np.float32), warped_hsi, scale_mat, sub)
            cubes.append(cube.astype(np.float32))
            continue

        if arr.ndim == 3:
            arr = arr.astype(np.float32)
            if arr.shape[-1] <= 64 and arr.shape[0] > 64 and arr.shape[1] > 64:
                arr = arr.transpose(2, 0, 1)
            arr = resize_chw(arr, sub, sub)
            cubes.append(arr)
            continue

        raise ValueError(f"不支持的 npy 形状: {arr.shape}，文件: {p}")

    cond = stitch_2x2_np(cubes)
    if cond.shape[0] != 48:
        raise ValueError(f"cond 通道数不是 48，实际为 {cond.shape[0]}，文件组: {paths}")
    return cond


# ================== 载入模型 ==================
controlnet = WarpAndControlNetModel.from_pretrained(
    CKPT_DIR,
    torch_dtype=DTYPE,
    use_safetensors=True,
    safety_checker=None,
)

controlnet = controlnet.to(device=DEVICE, dtype=DTYPE)
if hasattr(controlnet, "warper") and controlnet.warper is not None:
    controlnet.warper = controlnet.warper.to(device=DEVICE, dtype=DTYPE)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    use_safetensors=True,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if DEVICE == "cuda":
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(DEVICE)


# ================== 分组推理 ==================
grouped = group_npy_files(NPY_DIR)
if len(grouped) == 0:
    raise RuntimeError("未找到可用分组：需要命名为 num1_num2_num3_num4，且每个 num123 都包含 num4=0/90/180/270")

print(f"共找到 {len(grouped)} 组 num123")
warped_hsi, scale_mat = init_hsi_and_warped(image_size=IMAGE_SIZE)

for i, (num123, paths) in enumerate(tqdm(grouped, desc="Inference from npy"), start=1):
    prompt = prompt_from_group_paths(paths)
    tqdm.write(f"[{i}/{len(grouped)}] num123={num123}, prompt={prompt}")

    cond48 = load_cond48_from_group(paths, warped_hsi, scale_mat, image_size=IMAGE_SIZE)
    cond48 = torch.from_numpy(cond48).unsqueeze(0).to(device=DEVICE, dtype=DTYPE)

    with torch.inference_mode():
        image = pipe(
            prompt=prompt,
            image=cond48,
            guidance_scale=3.0,
        ).images[0]

    save_path = os.path.join(OUT_DIR, f"num123_{num123}_{i:04d}.png")
    image.save(save_path)

print(f"✅ 完成，输出目录: {OUT_DIR}")

Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-21500/controlnet/warper.pt


Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  3.31it/s]


共找到 9 组 num123


Inference from npy:   0%|          | 0/9 [00:00<?, ?it/s]

[1/9] num123=0_0_0, prompt=city topview from camera, camera conf as following. {camera_height:173.205078125,tilt_deg:60.0,lens:100.0}


...

[8/9] num123=7_0_0, prompt=city topview from camera, camera conf as following. {camera_height:173.205078125,tilt_deg:60.0,lens:100.0}


Inference from npy:  89%|████████▉ | 8/9 [01:14<00:09,  9.57s/it]

[9/9] num123=8_0_0, prompt=city topview from camera, camera conf as following. {camera_height:173.205078125,tilt_deg:60.0,lens:100.0}


Inference from npy: 100%|██████████| 9/9 [01:23<00:00,  9.29s/it]

✅ 完成，输出目录: ./results_grid4
